# llm_local · 01 — Generazione del dataset delle sintesi

Questo notebook crea gli esempi di addestramento per il modello piccolo.

Per ogni esempio:
1. estrae **3 arcani maggiori** diversi (passato, presente, futuro), ognuno **dritto o rovesciato** a caso;
2. prende le tue descrizioni da `descrizione_it`;
3. fa scrivere a un modello grande (via **Ollama**) la **sintesi finale** della lettura (circa 1 minuto letta ad alta voce), da ascoltare dopo i video delle singole carte;
4. salva tutto in `/kaggle/working/letture.jsonl`.

### Prima di avviare (pannello a destra, *Settings*)
- **Accelerator**: `GPU P100` oppure `GPU T4 x2`
- **Internet**: `On` (serve per scaricare Ollama e il modello)
- **Input**: aggiungi il tuo dataset con la cartella `descrizione_it`

Il notebook riprende da dove si era fermato: se la sessione si interrompe, rilancialo e continua.

In [ ]:
# ===== IMPOSTAZIONI =====
MODELLO     = "gemma3:12b"   # alternativa più veloce: "qwen2.5:7b"
N_LETTURE   = 2000           # quante letture generare in totale
N_PARALLELO = 4              # richieste contemporanee a Ollama
SEED        = 42
OUTPUT      = "/kaggle/working/letture.jsonl"

from datetime import datetime
from zoneinfo import ZoneInfo
def fatto(cella):
    print(f"\n✅ Completata: {cella} — ore {datetime.now(ZoneInfo('Europe/Rome')):%H:%M:%S}")

fatto("Impostazioni")

## 1. Installo e avvio Ollama

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

fatto("Installazione Ollama")

In [ ]:
import os, subprocess, time, requests

env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434",
           OLLAMA_NUM_PARALLEL=str(N_PARALLELO), OLLAMA_KEEP_ALIVE="24h")
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2); print("Ollama è attivo"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama non parte: guarda /kaggle/working/ollama.log")

!ollama pull {MODELLO}
!nvidia-smi --query-gpu=name,memory.total --format=csv

fatto("Avvio Ollama e download modello")

## 2. Carico le descrizioni delle carte

In [ ]:
import glob, re

trovati = glob.glob("/kaggle/input/**/00_il_matto_Passato.md", recursive=True)
assert trovati, "Non trovo descrizione_it: aggiungi il dataset nel pannello Input"
CARTELLA = os.path.dirname(trovati[0])
print("Descrizioni in:", CARTELLA)

NOMI = {
    "00_il_matto": "Il Matto", "01_il_mago": "Il Mago", "02_la_papessa": "La Papessa",
    "03_l_imperatrice": "L'Imperatrice", "04_l_imperatore": "L'Imperatore", "05_il_papa": "Il Papa",
    "06_gli_amanti": "Gli Amanti", "07_il_carro": "Il Carro", "08_la_forza": "La Forza",
    "09_l_eremita": "L'Eremita", "10_la_ruota_della_fortuna": "La Ruota della Fortuna",
    "11_la_giustizia": "La Giustizia", "12_l_appeso": "L'Appeso", "13_la_morte": "La Morte",
    "14_la_temperanza": "La Temperanza", "15_il_diavolo": "Il Diavolo", "16_la_torre": "La Torre",
    "17_la_stella": "La Stella", "18_la_luna": "La Luna", "19_il_sole": "Il Sole",
    "20_il_giudizio": "Il Giudizio", "21_il_mondo": "Il Mondo",
}
POSIZIONI = ["Passato", "Presente", "Futuro"]

def pulisci(testo):
    testo = testo.replace("\r\n", "\n").replace("\u2019", "'")
    righe = [r.strip() for r in testo.split("\n")]
    return "\n".join(r for r in righe if r)

DESCR = {}
for slug in NOMI:
    for rov in (False, True):
        for pos in POSIZIONI:
            nome_file = f"{slug}_{'r_' if rov else ''}{pos}.md"
            with open(os.path.join(CARTELLA, nome_file), encoding="utf-8") as f:
                DESCR[(slug, rov, pos)] = pulisci(f.read())

print(len(DESCR), "descrizioni caricate (attese 132)")
assert len(DESCR) == 132

fatto("Caricamento descrizioni")

## 3. Estrazione e prompt

In [ ]:
import random, json

# La sintesi viene ascoltata DOPO i tre video delle singole carte:
# non deve ripeterli, deve collegarli.
SYSTEM_GENERATORE = (
    "Sei un esperto lettore di tarocchi. La persona ha appena ascoltato la spiegazione di tre carte "
    "(passato, presente, futuro). Tu scrivi la sintesi finale della lettura, che verrà letta ad alta voce.\n"
    "Regole:\n"
    "- Non ripetere le singole descrizioni: la persona le ha già ascoltate. Usale solo per capire il significato.\n"
    "- Cita le tre carte per nome, nell'ordine passato, presente, futuro, senza rispiegarle.\n"
    "- Davanti ai nomi delle carte usa l'italiano naturale: articolo minuscolo a metà frase e preposizione "
    "unita all'articolo quando serve. Nomina solo le tre carte uscite, mai altre.\n"
    "- Linguaggio neutro: la persona può essere un uomo o una donna. Non usare aggettivi o participi "
    "riferiti a lei con il maschile o il femminile (per esempio non scrivere sei bloccato o ti senti libera): "
    "riformula con verbi e sostantivi, come hai fatto fatica a lasciare andare, senti il bisogno di libertà, fidati di te.\n"
    "- Il cuore della sintesi è il collegamento: quale filo unisce le tre carte, come il passato ha portato "
    "al presente e verso dove si muove il futuro. Tieni conto se una carta è rovesciata.\n"
    "- Dai del tu. Non parlare in prima persona e non fingere di conoscere la vita della persona.\n"
    "- Non inventare dettagli concreti (luoghi, oggetti, persone, episodi) e non fare previsioni "
    "su salute, morte o eventi certi.\n"
    "- Tra 120 e 160 parole, in uno o due paragrafi.\n"
    "- Scrivi per l'ascolto: frasi scorrevoli e non troppo lunghe, niente parentesi, elenchi, simboli, "
    "titoli o markdown. Niente frasi come 'Ecco la sintesi' o 'In sintesi'."
)

# Variano da una sintesi all'altra, così gli esempi non si somigliano tutti
ATTACCHI = [
    "Apri nominando il tema che lega le tre carte.",
    "Apri con un'immagine che rappresenti il percorso, poi nomina le carte.",
    "Apri con una domanda rivolta alla persona, poi collega le carte.",
    "Apri partendo dalla carta del passato e da cosa ha messo in moto.",
    "Apri in modo semplice e diretto, mettendo in fila le tre carte uscite.",
    "Apri sottolineando il contrasto o l'armonia tra le tre carte.",
]
CHIUSURE = [
    "Chiudi con un consiglio pratico per i prossimi tempi.",
    "Chiudi con una frase breve e incoraggiante.",
    "Chiudi indicando su cosa concentrare le energie.",
    "Chiudi riprendendo l'immagine o il tema dell'apertura.",
    "Chiudi con un invito alla riflessione, senza dare ordini.",
]

# Istruzione breve che vedrà il modello piccolo durante il fine-tuning
SYSTEM_STUDENTE = (
    "Sei un lettore di tarocchi. Scrivi in italiano la sintesi finale della lettura delle tre carte, "
    "dando del tu, da leggere ad alta voce in circa un minuto."
)

# Parola chiave per riconoscere ogni carta nel testo (es. "La Ruota della Fortuna" -> "ruota")
def _parola_chiave(nome):
    return re.sub(r"^(il |la |l'|gli )", "", nome.lower()).split()[0]
CHIAVE_NOME = {n: _parola_chiave(n) for n in NOMI.values()}
CARTE_CAPITALIZZATE = {n: CHIAVE_NOME[n].capitalize() for n in NOMI.values()}

# Aggettivi e participi al maschile o femminile riferiti alla persona
GENERE = (r"\b(?:sei|eri|sarai|ti senti|ti sentivi|ti sei sentit[oa]|ti trovi|ti ritrovi|rimani|resti|ti mostri)"
          r"\s+(?:\w+\s+)?\w+(?:at|ut|it|ss|ent|os|iv|ol|er|ur)[oa]\b"
          r"|\b(?:intrappolat|ancorat|bloccat|imprigionat|appagat|smarrit|confus|delus|sereno|serena|pronto|pronta|stanco|stanca)[oa]?\b"
          r"|\bte stess[oa]\b|\bda sol[oa]\b")

def estrai(rng):
    carte = rng.sample(list(NOMI), 3)
    return [{"slug": s, "nome": NOMI[s], "rovesciata": rng.random() < 0.5, "posizione": p}
            for s, p in zip(carte, POSIZIONI)]

def chiave(estrazione):
    return "|".join(f"{c['slug']}{'_r' if c['rovesciata'] else ''}" for c in estrazione)

def messaggio_utente(estrazione):
    blocchi = []
    for c in estrazione:
        orient = "rovesciata" if c["rovesciata"] else "dritta"
        testo = DESCR[(c["slug"], c["rovesciata"], c["posizione"])]
        blocchi.append(f"{c['posizione'].upper()}: {c['nome']} (carta {orient})\n{testo}")
    return "\n\n".join(blocchi)

def normalizza(testo):
    testo = testo.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    testo = re.sub(r"[ \t]+\n", "\n", testo)
    return re.sub(r"\n{3,}", "\n\n", testo).strip()

def valida(testo, estrazione):
    parole = len(testo.split())
    if parole < 100 or parole > 190:
        return f"lunghezza {parole} parole"
    if re.search(r"[#*_()\[\]]|^\s*[-•]\s", testo, re.M):
        return "contiene markdown, parentesi o elenchi"
    if re.search(r"[,;:]\s*[.!?]|\s[,.;:!?]|\.\s*\.", testo):
        return "punteggiatura rotta o frase troncata"
    if re.match(r"\s*(ricordo|io |vedo|sento|ecco|in sintesi|riassumendo)", testo, re.I):
        return "attacco non valido"
    if re.search(r"\b(?:da|di|a|in|su|con|per|tra|fra)\s+(?:Il|La|Lo|L'|Gli)\b", testo):
        return "preposizione non articolata davanti a una carta"
    if re.search(GENERE, testo, re.I):
        return "aggettivo con genere: " + re.search(GENERE, testo, re.I).group(0)
    uscite = {c["nome"] for c in estrazione}
    for c in estrazione:
        if CHIAVE_NOME[c["nome"]] not in testo.lower():
            return f"manca la carta {c['nome']}"
    for nome, parola in CARTE_CAPITALIZZATE.items():
        if nome not in uscite and re.search(rf"\b(?i:il|la|lo|l'|gli|dal|dalla|dall'|del|della|dell'|nel|nella|nell'|al|alla|all'|sul|sulla|col|con)\s*{parola}\b", testo):
            return f"cita una carta non uscita: {nome}"
    return None

def genera(estrazione, tentativi=4, rng=random):
    utente = messaggio_utente(estrazione)
    errore = None
    for _ in range(tentativi):
        guida = f"{rng.choice(ATTACCHI)} {rng.choice(CHIUSURE)}"
        r = requests.post("http://127.0.0.1:11434/api/chat", timeout=600, json={
            "model": MODELLO, "stream": False,
            "messages": [{"role": "system", "content": SYSTEM_GENERATORE},
                         {"role": "user", "content": f"{utente}\n\nIndicazioni di stile: {guida}"}],
            "options": {"temperature": 0.8, "top_p": 0.95, "num_ctx": 4096, "num_predict": 400},
        })
        r.raise_for_status()
        testo = normalizza(r.json()["message"]["content"])
        errore = valida(testo, estrazione)
        if errore is None:
            return {
                "chiave": chiave(estrazione),
                "carte": estrazione,
                "messages": [
                    {"role": "system", "content": SYSTEM_STUDENTE},
                    {"role": "user", "content": utente},   # lo studente NON vede le indicazioni di stile
                    {"role": "assistant", "content": testo},
                ],
            }
    raise ValueError(f"scartata dopo {tentativi} tentativi: {errore}")

fatto("Prompt e funzioni")

## 4. Prova con 2 sintesi
Controlla che il risultato ti piaccia prima di lanciare tutto.

In [ ]:
rng_prova = random.Random(0)
for _ in range(2):
    e = estrai(rng_prova)
    t0 = time.time()
    es = genera(e)
    print(f"--- {chiave(e)}  ({time.time()-t0:.0f} s)\n")
    print(es["messages"][2]["content"], "\n")

fatto("Prova con 2 sintesi")

## 5. Generazione completa
Riprende automaticamente se `letture.jsonl` esiste già.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

fatte = set()
if os.path.exists(OUTPUT):
    with open(OUTPUT, encoding="utf-8") as f:
        fatte = {json.loads(r)["chiave"] for r in f if r.strip()}
print("Già presenti:", len(fatte))

rng = random.Random(SEED)
da_fare, viste = [], set(fatte)
while len(fatte) + len(da_fare) < N_LETTURE:
    e = estrai(rng)
    k = chiave(e)
    if k not in viste:
        viste.add(k); da_fare.append(e)

lock, scartate, t0 = threading.Lock(), 0, time.time()
with open(OUTPUT, "a", encoding="utf-8") as out, ThreadPoolExecutor(N_PARALLELO) as pool:
    futuri = [pool.submit(genera, e) for e in da_fare]
    for i, fut in enumerate(as_completed(futuri), 1):
        try:
            es = fut.result()
            with lock:
                out.write(json.dumps(es, ensure_ascii=False) + "\n"); out.flush()
        except Exception as ex:
            scartate += 1
        if i % 25 == 0 or i == len(futuri):
            min_tot = (time.time() - t0) / 60
            print(f"{i}/{len(futuri)}  scartate: {scartate}  "
                  f"tempo: {min_tot:.0f} min  stima residua: {min_tot / i * (len(futuri) - i):.0f} min")

fatto("Generazione completa")

## 6. Controllo finale
Il file `letture.jsonl` lo trovi nella sezione **Output** a destra: scaricalo e mettilo in `llm_local/data/`.

In [ ]:
with open(OUTPUT, encoding="utf-8") as f:
    righe = [json.loads(r) for r in f if r.strip()]
parole = [len(r["messages"][2]["content"].split()) for r in righe]
print(f"Letture: {len(righe)}")
print(f"Parole per lettura: min {min(parole)}, media {sum(parole)//len(parole)}, max {max(parole)}")
print("\nEsempio a caso:\n")
print(random.choice(righe)["messages"][2]["content"])

fatto("Controllo finale")